# Crash Course — 바이오메디컬 AI와 생성형 AI

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [crash-course/ai-biomed.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/crash-course/ai-biomed.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 최소 실행 예시

합성 표 데이터의 유용성과 프라이버시를 함께 재는 가장 짧은 코드다. 가상 데이터를 만들고, 성분 수가 적은 합성기와 과적합 합성기를 비교한다.

In [ ]:
import numpy as np, pandas as pd
from scipy import stats
from sklearn.mixture import GaussianMixture
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(7)
COLS = ["age", "sbp", "marker"]
n = 800
age = rng.normal(64, 9, n)
sbp = 95 + 0.6 * age + rng.normal(0, 11, n)
marker = 0.03 * age + rng.normal(0, 1, n)
y = (rng.random(n) < 1 / (1 + np.exp(-(-9 + 0.05 * age + 0.02 * sbp + 0.8 * marker)))).astype(int)
real = pd.DataFrame({"age": age, "sbp": sbp, "marker": marker, "y": y})
train, hold = train_test_split(real, test_size=0.5, random_state=0, stratify=real.y)

def synth(df, k):
    out = []
    for lab, g in df.groupby("y"):
        gm = GaussianMixture(n_components=min(k, len(g)), covariance_type="full", random_state=0).fit(g[COLS])
        s, _ = gm.sample(len(g))
        out.append(pd.DataFrame(s, columns=COLS).assign(y=lab))
    return pd.concat(out, ignore_index=True)

trtr = roc_auc_score(hold.y, LogisticRegression(max_iter=1000).fit(train[COLS], train.y).predict_proba(hold[COLS])[:, 1])
print(f"실제 학습 → 실제 평가(TRTR) AUC {trtr:.3f}")
sc = StandardScaler().fit(train[COLS])
A, H = sc.transform(train[COLS]), sc.transform(hold[COLS])
dcr_h = np.median(np.sqrt(((H[:, None] - A[None]) ** 2).sum(-1)).min(1))
for k in (2, 200):
    s = synth(train, k)
    tstr = roc_auc_score(hold.y, LogisticRegression(max_iter=1000).fit(s[COLS], s.y).predict_proba(hold[COLS])[:, 1])
    ks = max(stats.ks_2samp(train[c], s[c]).statistic for c in COLS)
    S = sc.transform(s[COLS])
    dcr = np.median(np.sqrt(((S[:, None] - A[None]) ** 2).sum(-1)).min(1))
    print(f"합성(성분 {k:>3}개): KS최대 {ks:.3f} | TSTR AUC {tstr:.3f} | DCR 합성 {dcr:.3f} vs 실제 holdout {dcr_h:.3f}")